# Learning a preferred writing style with DPO

We use fables from George Fyler Townsend’s translation of Aesop. The preparation manifest identifies the corpus; we read its actual split sizes and provenance below.

Our goal is **a bold headline followed by a short, faithful summary**. The preference pairs are synthetic and AI-authored—not collected human judgments. We will inspect a training pair, train the model, and measure the style of its actual held-out outputs.

## DPO, through one example

As an illustrative example (not a measured training result), imagine a passage in which a fisherman catches large fish while small fish escape through his net. 

> We prefer this presentation:

---
```text
**Small Fish Escape the Net**

A fisherman’s net retained the large fish, while smaller fish slipped through its mesh.
```

---


> The alternative has **exactly the same words**, but an unbolded headline:

---

```text
Small Fish Escape the Net

A fisherman’s net retained the large fish, while smaller fish slipped through its mesh.
```
---
The difference is a writing preference, not a factual correction. For this exercise the preferred response emits Markdown `**Headline**`, a blank line, and a summary. Other comparisons use the same summary without a headline.

**Direct Preference Optimization (DPO)** learns from these chosen/rejected comparisons. It adjusts the policy so the chosen answer gains relative likelihood over the alternative, **compared with a fixed starting reference**. The reference anchors that change; it is not an answer key. The beta parameter controls the reference-relative scaling in the DPO loss.

In classic RLHF, comparisons typically train a separate reward model, followed by an RL policy update such as PPO. Here, DPO directly updates the policy from pairs: **no separately trained reward model, no RL critic, and no PPO loop**. Our starting checkpoint is already instruction-tuned, so we are not repeating SFT.

```text
Classic RLHF (PPO): SFT policy -> comparisons -> reward model -> RL optimization
DPO:     instruction-tuned policy + comparisons + fixed reference -> direct policy updates
```

Importantly, the prompt simply asks for a faithful summary. It does **not** request headlines or bold text. We use that same neutral prompt before and after training, so formatting is not supplied by changing the evaluation instructions.

## 1. Setup and settings

The `short` profile checks training mechanics; `full` provides a longer style-learning experiment. Their optimizer-update budgets come from `dpo_utils.PROFILE_STEPS`. Both use the same prepared data. After loading it, we report actual split sizes and nominal training passes rather than assuming a fixed corpus size. Budgets are fixed before held-out inspection, not selected from test results.

`DEVICE='auto'` chooses an available accelerator or CPU. Set profile and resource settings in the next cell; the pinned checkpoint, beta, and default sequence limit are defined in `dpo_utils.py`. Short mode reduces update time, not model memory requirements, and does not establish learned style quality.

Run from this notebook's directory and select a Python kernel with dependencies from `requirements.txt` installed. Optional environment variables `DPO_PROFILE`, `DPO_DEVICE`, and `DPO_ARTIFACT_ROOT` support command-line runs. Every run gets a separate artifact directory; restart and run all cells for a fresh model.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, uuid
import json, math, time, platform, importlib.metadata as metadata
import warnings
warnings.filterwarnings('ignore', message='IProgress not found.*')  # optional widgets only
from html import escape
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from transformers import AutoModelForCausalLM, set_seed
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig, PeftModel
import dpo_utils as ru

# Both profiles share the same corpus and evaluation budget.
PROFILE = os.environ.get('DPO_PROFILE', 'full')  # 'short' or 'full'
DEVICE = os.environ.get('DPO_DEVICE', 'auto')   # 'auto', 'cpu', 'mps', or 'cuda'
SEED = 42
BETA = ru.BETA
MAX_LENGTH = ru.MAX_LENGTH
MAX_NEW_TOKENS = 128
MICROBATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
PREVIEW_COUNT = 4
CPU_THREADS = 4
STEPS = ru.PROFILE_STEPS
if PROFILE not in STEPS:
    raise ValueError(f'Unknown profile {PROFILE!r}; choose one of {list(STEPS)}')
torch.set_num_threads(CPU_THREADS)
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / 'data'
device = ru.choose_device(DEVICE)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
artifact_root = Path(os.environ.get('DPO_ARTIFACT_ROOT', str(PROJECT_DIR / 'artifacts')))
artifact_dir = artifact_root / f'{device}-{PROFILE}' / run_id
artifact_dir.mkdir(parents=True, exist_ok=False)
print('Device:', device, '| profile:', PROFILE, '| optimizer steps:', STEPS[PROFILE])
print('Artifacts:', artifact_dir)


## 2. Preference data

Each record supplies a passage, a chosen response, and a rejected response. Every pair retains the **same summary body**. Half compare bold versus unbolded headlines; half compare headline-plus-summary versus summary alone. The first contrast isolates the Markdown formatting preference, while the second teaches the broader presentation.

The prepared train/validation/test splits keep whole fables separate. We verify their counts and checksums against the manifest rather than assuming fixed sizes. Reasons and essential-fact notes are inspection metadata, **not model inputs**. Added headings/markup change token lengths; this is a deliberate style experiment, not a length-balanced factuality benchmark.

> From this notebook's directory, run these commands in your terminal. Raw fables are downloaded; summaries, headlines, and fact notes come from the bundled `source/authored_annotations.json` (AI-authored annotations, not collected human feedback). The second command deterministically builds all notebook data files without model/API calls.
```bash
python prepare_source.py --download
python prepare_data.py
```

In [ ]:
splits, data_provenance = ru.load_data_bundle(DATA_DIR)
counts = {name: len(rows) for name, rows in splits.items()}
test_rows = splits['test']
data_manifest = data_provenance['manifest']
print('Preference pairs:', counts, '| total:', sum(counts.values()))
print('Preference format:', data_manifest.get('format', 'headline-summary preferences'))
print('Label provenance:', data_provenance['label_origins'])
print('Annotation authors:', data_provenance['author_models'])
print('Manifest SHA-256:', data_provenance['manifest_sha256'])
if data_manifest.get('source_metadata'):
    print('Source:', data_manifest['source_metadata'])
print('Unique source groups:', {name: len({r['group_id'] for r in rows}) for name, rows in splits.items()})
from collections import Counter
for name, rows in splits.items():
    print(name, 'contrasts:', dict(Counter(r['rejection_type'] for r in rows)))
effective_batch_size = MICROBATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
display(pd.DataFrame([
    {'profile': name, 'optimizer_updates': steps,
     'nominal_pair_exposures': steps * effective_batch_size,
     'nominal_training_passes': steps * effective_batch_size / counts['train']}
    for name, steps in STEPS.items()
]))
print('Nominal budgets assume one device and complete accumulation batches; no held-out tuning.')


### One actual training example

Read the passage, then compare the two candidate answers. In this example the wording is identical and only the bold markup differs. The ranking encodes our selected presentation style; it does not make one version more truthful.

In [ ]:
example = next(r for r in splits['train'] if r['rejection_type'] == 'unbolded_headline')
display(Markdown('**Passage (included in the prompt)**\n\n' + escape(example['passage'])))
display(Markdown('**Chosen response**'))
display(Markdown(escape(example['chosen'])))
display(Markdown('**Rejected response**'))
display(Markdown(escape(example['rejected'])))
print('Raw chosen text, including Markdown and paragraph breaks:')
print(repr(example['chosen']))
print('Why preferred:', example['preference_reason'])
print('Label provenance:', example['label_origin'])

## 3. Model and tokenization

We use a pinned small instruction-tuned checkpoint. Only source passages and the neutral summarization request enter its prompt—not source titles, preferred answers, or reasons. The helper checks complete prompt/response tokenization, EOS, and sequence length. It refuses overlong examples rather than silently dropping facts or response tokens.

In [ ]:
set_seed(SEED)
base, tokenizer = ru.load_base(seed=SEED, num_threads=CPU_THREADS)
train = ru.make_dataset(splits['train'], tokenizer, max_length=MAX_LENGTH)
validation = ru.make_dataset(splits['validation'], tokenizer, max_length=MAX_LENGTH)
_ = ru.make_dataset(splits['test'], tokenizer, max_length=MAX_LENGTH)
lengths = [len(ru.validate_and_tokenize(tokenizer, ru.render_prompt(tokenizer, r['passage']),
                                      r[side], MAX_LENGTH)[0])
           for rows in splits.values() for r in rows for side in ('chosen', 'rejected')]
print('Pinned model:', ru.MODEL_ID, '@', ru.MODEL_REVISION)
print('Actual paired sequence lengths:', min(lengths), 'to', max(lengths), '| allowed:', MAX_LENGTH)
print('Training columns:', train.column_names)
print('Actual prompt for the example (no style instructions):')
print(ru.render_prompt(tokenizer, example['passage']))

## 4. Configure DPO and LoRA

> **LoRA, briefly:** LoRA makes fine-tuning more economical by training small additional parameter matrices instead of updating every model weight. For now, think of it as a resource-saving training technique; later chapters explain how it works. It is not a replacement optimizer.

The base weights remain frozen; only LoRA adapters train. Disabling the adapter gives the fixed starting reference without keeping a second full model copy. Full mode precomputes reference log probabilities. Evaluation is manual before/after; it does not select checkpoints or stop training early.

In [ ]:
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0,
                   target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM')
monitor = ru.TrainingMonitor()
config = DPOConfig(
    output_dir=str(artifact_dir / 'trainer'), learning_rate=LEARNING_RATE,
    per_device_train_batch_size=MICROBATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_steps=STEPS[PROFILE], per_device_eval_batch_size=1,
    eval_strategy='no', save_strategy='no', report_to='none', logging_steps=1,
    disable_tqdm=True, use_cpu=(device == 'cpu'), bf16=False, fp16=False,
    max_length=MAX_LENGTH, truncation_mode='keep_start', loss_type='sigmoid', beta=BETA,
    precompute_ref_log_probs=(PROFILE == 'full'), precompute_ref_batch_size=1,
    seed=SEED, data_seed=SEED, gradient_checkpointing=False, dataloader_pin_memory=False)
trainer = DPOTrainer(model=base, ref_model=None, args=config,
    processing_class=tokenizer, train_dataset=train, eval_dataset=validation,
    peft_config=lora, callbacks=[monitor])
policy = trainer.model
actual_device = next(policy.parameters()).device
assert actual_device.type == device
assert all('lora_' in name for name, p in policy.named_parameters() if p.requires_grad)
trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
total = sum(p.numel() for p in policy.parameters())
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)')

## 5. Measure the starting model

Creating the trainer is not training. Now that the policy is on its actual device, we score all loaded test pairs and generate one output for every test fable **before any update**. Initially policy and reference coincide: DPO margins are zero and loss is `log(2)`. Our reference-relative preferred fraction gives ties half-credit (0.5); TRL's strict reward accuracy gives ties zero. Neither is a factual-quality score.

The separate style checks inspect actual generated text: a standalone Markdown-bold first line, and the complete headline/blank-line/body structure. Bold words buried in prose or a fenced code block do not count as the target headline.

In [ ]:
validation_before = trainer.evaluate(eval_dataset=validation)
test_before_scores = ru.score_pairs(policy, tokenizer, test_rows, beta=BETA, max_length=MAX_LENGTH)
test_before_summary = ru.summarize_pairs(test_before_scores)
test_before_generations = ru.generate_summaries(policy, tokenizer, test_rows,
    max_new_tokens=MAX_NEW_TOKENS, reference=True)
style_before_scores = ru.score_styles(test_before_generations)
style_before_summary = ru.summarize_styles(style_before_scores)
assert abs(test_before_summary['mean_dpo_loss'] - math.log(2)) < 1e-5
assert test_before_summary['tie_fraction'] == 1.0
print('Before, synthetic pair statistics:', test_before_summary)
print('Before, generated style:', style_before_summary)

## 6. Train and verify real updates

An optimizer hook counts actual updates, not gradient-accumulation microbatches. A sampled base-weight value check and changed LoRA-B tensors provide additional guards. They verify mechanics, not answer quality.

In [ ]:
base_snapshot = ru.BaseWeightSnapshot(policy)
lora_b_before = {name: p.detach().cpu().clone() for name, p in policy.named_parameters()
                 if 'lora_b' in name.lower()}
assert lora_b_before
trainer.create_optimizer()
step_counter = ru.OptimizerStepCounter(trainer.optimizer)
set_seed(SEED)
start = time.perf_counter()
train_result = trainer.train()
training_seconds = time.perf_counter() - start
optimizer_steps = step_counter.close()
assert optimizer_steps == STEPS[PROFILE] == trainer.state.global_step
assert base_snapshot.unchanged(policy)
changed_lora_b = [name for name, p in policy.named_parameters()
                  if name in lora_b_before and not torch.equal(lora_b_before[name], p.detach().cpu())]
assert changed_lora_b
assert all(math.isfinite(float(v)) for row in monitor.history for v in row.values() if isinstance(v, (int, float)))
print('Actual optimizer updates:', optimizer_steps, '| training seconds:', round(training_seconds, 2))
print('Frozen base sample unchanged:', base_snapshot.unchanged(policy), '| changed LoRA-B tensors:', len(changed_lora_b))

## 7. Measure after DPO

Use exactly the same prompts and deterministic decoding budget. Reference scores remain those from the fixed starting model. These diagnostics compare synthetic preferences and presentation style, not truth, safety, or human satisfaction. We retain every test result, including failures.

In [ ]:
validation_after = trainer.evaluate(eval_dataset=validation)
test_after_scores = ru.score_pairs(policy, tokenizer, test_rows, beta=BETA,
    reference_scores={r['id']: r for r in test_before_scores}, max_length=MAX_LENGTH)
test_after_summary = ru.summarize_pairs(test_after_scores)
test_after_generations = ru.generate_summaries(policy, tokenizer, test_rows, max_new_tokens=MAX_NEW_TOKENS)
style_after_scores = ru.score_styles(test_after_generations)
style_after_summary = ru.summarize_styles(style_after_scores)
display(pd.DataFrame([{'stage': 'before', **style_before_summary}, {'stage': 'after', **style_after_summary}]))
display(pd.DataFrame([{'stage': 'before', **test_before_summary}, {'stage': 'after', **test_after_summary}]))
print('Validation DPO loss:', validation_before['eval_loss'], '->', validation_after['eval_loss'])

### Before/after style and preference plots

The first two panels measure generated formatting; the last two measure the synthetic pair objective. The fraction is reference-relative and gives initial ties half-credit—it does not mean half the summaries were correct. Each percentage is based on the actual held-out sample count shown on the axes; this small demonstration is not a statistical quality benchmark.

In [ ]:
stages = ['Before', 'After']
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
plots = [
    ('Bold headline', [style_before_summary['bold_headline_rate'], style_after_summary['bold_headline_rate']], True),
    ('Headline + summary format', [style_before_summary['headline_summary_format_rate'], style_after_summary['headline_summary_format_rate']], True),
    ('Reference-relative preference', [test_before_summary['preferred_fraction'], test_after_summary['preferred_fraction']], True),
    ('Held-out DPO loss', [test_before_summary['mean_dpo_loss'], test_after_summary['mean_dpo_loss']], False)]
for ax, (title, values, percent) in zip(axes, plots):
    shown = [100*v for v in values] if percent else values
    bars = ax.bar(stages, shown, color=['#4C78A8', '#F58518'])
    ax.bar_label(bars, labels=[f'{v:.0f}%' if percent else f'{v:.3f}' for v in shown], padding=3)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(f'Percent (n={counts["test"]})' if percent else 'Loss (lower is better)')
    ax.set_ylim(0, 110 if percent else max(shown)*1.25)
    ax.grid(axis='y', alpha=.25)
fig.suptitle('Measured style and synthetic preference changes — not factual accuracy')
fig.tight_layout()
fig.savefig(artifact_dir / 'style_before_after.png', dpi=140, bbox_inches='tight')
plt.show()

## 8. Inspect actual text, not only metrics

These are the first test fables in fixed order, up to `PREVIEW_COUNT`, not examples selected for success. The model's Markdown is rendered below so you can see the headline styling; the raw text is also printed. Compare against the source and AI-authored essential-fact notes. A perfectly formatted answer can still invent events.

In [ ]:
before_by_id = {r['id']: r for r in test_before_generations}
after_by_id = {r['id']: r for r in test_after_generations}
for row in test_rows[:PREVIEW_COUNT]:
    display(Markdown('### ' + escape(row['title'])))
    display(Markdown('**Source:** ' + escape(row['passage'])))
    print('Essential facts (AI rubric):', row['essential_facts'])
    display(Markdown('**Before DPO:**'))
    display(Markdown(escape(before_by_id[row['id']]['summary'])))
    display(Markdown('**After DPO:**'))
    text = after_by_id[row['id']]['summary']
    display(Markdown(escape(text)))
    print('Raw after text:', repr(text))
    print('Style checks:', ru.style_flags(text))

## 9. Training telemetry

Loss and reward margins diagnose optimization over training pairs. They are not independent evaluations of generated summaries. Keep these traces alongside the separate held-out before/after plot.

In [ ]:
history = monitor.history
loss_points = [(r['step'], r['loss']) for r in history if 'loss' in r]
margin_points = [(r['step'], r['rewards/margins']) for r in history if 'rewards/margins' in r]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, points, title in zip(axes, [loss_points, margin_points], ['Trainer loss', 'Trainer reward margin']):
    if points: ax.plot(*zip(*points), marker='.', linewidth=1)
    ax.set_title(title if points else 'No logged ' + title.lower())
    ax.set_xlabel('Optimizer step'); ax.grid(True, alpha=.3)
fig.tight_layout()
fig.savefig(artifact_dir / 'telemetry.png', dpi=140, bbox_inches='tight')
plt.show()
print('Telemetry entries:', len(history))

## 10. Save and reload

A saved LoRA adapter is not a standalone model: reload it with the same pinned base checkpoint. Identical deterministic greedy token IDs on a checked held-out prompt verify serialization/inference consistency, not correctness.

In [ ]:
adapter_dir = artifact_dir / 'adapter'
policy.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
reload_base = AutoModelForCausalLM.from_pretrained(ru.MODEL_ID, revision=ru.MODEL_REVISION,
    trust_remote_code=False, dtype=torch.float32, attn_implementation='eager')
reload_base.config.use_cache = False
reloaded = PeftModel.from_pretrained(reload_base, adapter_dir).to(actual_device)
reloaded.eval()
row = test_rows[0]
prompt = ru.render_prompt(tokenizer, row['passage'])
inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(actual_device)
def greedy_ids(model):
    with torch.inference_mode():
        return model.generate(**inputs, do_sample=False, use_cache=True, max_new_tokens=MAX_NEW_TOKENS,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)[0].tolist()
policy.eval()
trained_ids = greedy_ids(policy)
reloaded_ids = greedy_ids(reloaded)
assert trained_ids == reloaded_ids
print('Reload: identical greedy token IDs for', row['id'])

## 11. Save all evaluation evidence

The JSON retains all before/after test generations, pair scores, style flags, summaries, validation diagnostics, version/runtime metadata, and mechanical checks—not just the displayed preview. The two saved plots are beside it. The record includes the verified data manifest and split hashes so results identify the exact corpus, plus experiment settings and the neutral prompt.

In [ ]:
def plain(value):
    if isinstance(value, dict): return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [plain(v) for v in value]
    if isinstance(value, set): return [plain(v) for v in sorted(value)]
    if hasattr(value, 'item'): return plain(value.item())
    if isinstance(value, (str, int, float, bool)) or value is None:
        if isinstance(value, float) and not math.isfinite(value): raise ValueError('Nonfinite JSON value')
        return value
    return str(value)
record = {
    'run_id': run_id, 'profile': PROFILE, 'device': device, 'seed': SEED, 'beta': BETA,
    'max_length': MAX_LENGTH, 'max_new_tokens': MAX_NEW_TOKENS,
    'paired_sequence_min_tokens': min(lengths), 'paired_sequence_max_tokens': max(lengths),
    'model_id': ru.MODEL_ID, 'model_revision': ru.MODEL_REVISION,
    'data_origin': data_manifest.get('source_metadata'),
    'data_provenance': data_provenance,
    'labels': data_provenance['label_origins'], 'split_counts': counts,
    'annotation_authors': data_provenance['author_models'],
    'profile_steps': dict(STEPS), 'microbatch_size': MICROBATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'learning_rate': LEARNING_RATE, 'preview_count': PREVIEW_COUNT,
    'cpu_threads': CPU_THREADS, 'prompt_example': ru.render_prompt(tokenizer, example['passage']),
    'dpo_config': config.to_dict(), 'lora_config': lora.to_dict(),
    'style_target': ru.STYLE_TARGET,
    'versions': {name: metadata.version(name) for name in ['torch','transformers','trl','peft','datasets','accelerate']},
    'python': platform.python_version(), 'training_seconds': training_seconds, 'optimizer_steps': optimizer_steps,
    'validation_before': plain(validation_before), 'validation_after': plain(validation_after),
    'test_before_summary': test_before_summary, 'test_after_summary': test_after_summary,
    'test_before_scores_all': test_before_scores, 'test_after_scores_all': test_after_scores,
    'test_before_generations_all': test_before_generations, 'test_after_generations_all': test_after_generations,
    'style_before_summary': style_before_summary, 'style_after_summary': style_after_summary,
    'style_before_scores_all': style_before_scores, 'style_after_scores_all': style_after_scores,
    'checks': {'base_sample_unchanged': base_snapshot.unchanged(policy), 'lora_b_changed': bool(changed_lora_b),
               'optimizer_steps_exact': optimizer_steps == STEPS[PROFILE], 'reload_greedy_ids_identical': trained_ids == reloaded_ids},
    'training_history': plain(history), 'adapter_note': 'Adapter requires the pinned base model revision.'}
record_path = artifact_dir / 'evaluation.json'
record_path.write_text(json.dumps(plain(record), ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
print('Saved:', record_path)

## What have we learned?

Compare the actual before/after style rates and text. DPO can teach a presentation preference without putting that preference in the prompt. But **bold headlines are not evidence of factual accuracy**. Check sources and human judgments before claiming better summarization.

The labels and fact notes here are AI-assisted, lengths differ by format, famous fables may occur in pretraining, and one seed with a small held-out set is a small demonstration. Short mode verifies mechanics only. None of these diagnostics certifies general alignment, safety, or human satisfaction.